In [ ]:
import sys, platform, random
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import seaborn as sns

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Python      :", sys.version.split()[0])
print("Platform    :", platform.platform())
print("numpy       :", np.__version__)
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("matplotlib  :", matplotlib.__version__)
print("seaborn     :", sns.__version__)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

In [45]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

ds = load_dataset("papluca/language-identification")
print(ds)

In [ ]:
import pandas as pd

df_train = ds["train"].to_pandas()
df_val   = ds["validation"].to_pandas()
df_test  = ds["test"].to_pandas()

print("Форма train:", df_train.shape)
print("Форма val  :", df_val.shape)
print("Форма test :", df_test.shape)
print()
print("Колонки:", df_train.columns.tolist())
print()
print("Первые 5 строк:")
df_train.head()

In [ ]:
print("Всего языков:", df_train["labels"].nunique())
print()
print("Распределение по языкам (train):")
print(df_train["labels"].value_counts())

In [ ]:
LANGS = ["en", "fr", "de", "es", "it", "ru", "pt", "nl"]

df_train = df_train[df_train["labels"].isin(LANGS)].reset_index(drop=True)
df_val   = df_val[df_val["labels"].isin(LANGS)].reset_index(drop=True)
df_test  = df_test[df_test["labels"].isin(LANGS)].reset_index(drop=True)

print("После фильтрации:")
print("train:", df_train.shape)
print("val  :", df_val.shape)
print("test :", df_test.shape)
print()
print("Распределение (train):")
print(df_train["labels"].value_counts())

In [ ]:
for lang in LANGS:
    sample = df_train[df_train["labels"] == lang]["text"].head(3).tolist()
    print(f"--- {lang} ---")
    for s in sample:
        print("  ", s)
    print()

In [ ]:
df_train["len_chars"] = df_train["text"].str.len()

print("Статистика длины (в символах) по всем языкам:")
print(df_train["len_chars"].describe())
print()
print("Медиана длины по языкам:")
print(df_train.groupby("labels")["len_chars"].median())

In [ ]:
df_train.to_csv("train_8langs.csv", index=False)
df_val.to_csv("val_8langs.csv", index=False)
df_test.to_csv("test_8langs.csv", index=False)
print("Сохранено. Файлы:", ["train_8langs.csv", "val_8langs.csv", "test_8langs.csv"])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, df) in zip(axes, [("train", df_train), ("val", df_val), ("test", df_test)]):
    counts = df["labels"].value_counts().reindex(LANGS)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="viridis")
    ax.set_title(f"{name} (n={len(df)})")
    ax.set_xlabel("язык")
    ax.set_ylabel("кол-во примеров")
    for i, v in enumerate(counts.values):
        ax.text(i, v + 20, str(v), ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("class_balance.png", dpi=120)
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_train, x="labels", y="len_chars", order=LANGS, palette="viridis")
plt.yscale("log")   # лог-шкала, потому что разброс от 5 до 2400
plt.axhline(20, color="red", linestyle="--", linewidth=1, label="порог «короткий текст» = 20 символов")
plt.axhline(10, color="orange", linestyle="--", linewidth=1, label="очень короткий = 10 символов")
plt.title("Длина текста (в символах) по языкам — log scale")
plt.ylabel("длина, символов")
plt.legend()
plt.tight_layout()
plt.savefig("length_by_lang.png", dpi=120)
plt.show()

# Сколько текстов короче 20 символов по языкам?
print("Доля текстов короче 20 символов, %:")
short_frac = (df_train.assign(is_short=df_train["len_chars"] < 20)
                        .groupby("labels")["is_short"].mean().reindex(LANGS) * 100)
print(short_frac.round(1))

In [ ]:
print("=== Самые короткие тексты (топ-3 на язык) ===")
for lang in LANGS:
    short = (df_train[df_train["labels"] == lang]
             .nsmallest(3, "len_chars")[["text", "len_chars"]])
    print(f"\n--- {lang} ---")
    for _, row in short.iterrows():
        print(f"  [{row['len_chars']:>3} симв.]  {row['text']!r}")

In [ ]:
from collections import Counter

def top_ngrams(texts, n=3, k=8):
    """Топ-k самых частых символьных n-грамм в наборе текстов."""
    c = Counter()
    for t in texts:
        t = t.lower()
        for i in range(len(t) - n + 1):
            c[t[i:i+n]] += 1
    return c.most_common(k)

print("Топ-8 символьных триграмм по языкам (без нормализации):")
for lang in LANGS:
    texts = df_train.loc[df_train["labels"] == lang, "text"].tolist()
    grams = top_ngrams(texts, n=3, k=8)
    pretty = ", ".join(f"{g!r}:{c}" for g, c in grams)
    print(f"  {lang}: {pretty}")

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score, classification_report
import time

baseline_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",     # n-граммы по символам с учётом границ слов
        ngram_range=(1, 3),     # уни-, би- и триграммы
        lowercase=True,
        sublinear_tf=True,      # log(1+tf) — стандарт для текста
        min_df=2,               # отбрасываем n-граммы, встречающиеся <2 раз
    )),
    ("clf", MultinomialNB()),
])

t0 = time.time()
baseline_pipe.fit(df_train["text"].values, df_train["labels"].values)
fit_time = time.time() - t0
print(f"Обучение заняло: {fit_time:.2f} сек")

In [ ]:
vec = baseline_pipe.named_steps["tfidf"]
print("Размер словаря n-грамм:", len(vec.vocabulary_))

In [ ]:
def evaluate(pipe, df, name=""):
    X, y = df["text"].values, df["labels"].values
    y_pred = pipe.predict(X)
    acc = accuracy_score(y, y_pred)
    f1m = f1_score(y, y_pred, average="macro")
    print(f"[{name}] accuracy={acc:.4f}  macro-F1={f1m:.4f}")
    return y_pred

print("Baseline (char_wb 1-3 + MNB):")
_ = evaluate(baseline_pipe, df_val, "val")
_ = evaluate(baseline_pipe, df_test, "test")

In [ ]:
y_pred_test = baseline_pipe.predict(df_test["text"].values)
print(classification_report(df_test["labels"], y_pred_test, digits=4))

In [ ]:
samples = [
    "hello world",
    "bonjour le monde",
    "привет мир",
    "hola amigo",
    "guten Tag",
    "ciao bella",
    "olá amigo",
    "goedemorgen",
    # провокации:
    "la casa",          # es или it?
    "il cane",          # it
    "o cão",            # pt
    "das Haus",         # de
    "the house",        # en
]
for s in samples:
    pred = baseline_pipe.predict([s])[0]
    print(f"  {s!r:25} -> {pred}")

In [ ]:
import itertools, time
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

X_tr, y_tr = df_train["text"].values, df_train["labels"].values
X_va, y_va = df_val["text"].values,   df_val["labels"].values

results = []
for ngram, min_df in itertools.product([(1,2), (1,3), (1,4), (2,4), (2,5)], [2, 3, 5]):
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=ngram,
                                  lowercase=True, sublinear_tf=True, min_df=min_df)),
        ("clf", MultinomialNB()),
    ])
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    fit_t = time.time() - t0
    y_pred = pipe.predict(X_va)
    acc = accuracy_score(y_va, y_pred)
    f1m = f1_score(y_va, y_pred, average="macro")
    n_feats = len(pipe.named_steps["tfidf"].vocabulary_)
    results.append(dict(ngram=str(ngram), min_df=min_df, n_features=n_feats,
                        val_acc=acc, val_f1=f1m, fit_sec=round(fit_t, 1)))

import pandas as pd
res_df = pd.DataFrame(results).sort_values("val_f1", ascending=False)
print(res_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

pivot = res_df.pivot(index="ngram", columns="min_df", values="val_f1")
# упорядочим ngram по возрастанию (n_min, n_max)
order = sorted(pivot.index, key=lambda s: eval(s))
pivot = pivot.loc[order]

plt.figure(figsize=(9, 5))
sns.heatmap(pivot, annot=True, fmt=".4f", cmap="YlGnBu", cbar_kws={"label": "val macro-F1"})
plt.title("Baseline MNB: macro-F1 vs ngram_range × min_df")
plt.xlabel("min_df")
plt.ylabel("ngram_range")
plt.tight_layout()
plt.savefig("grid_ngram_min_df.png", dpi=120)
plt.show()

In [ ]:
import time
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

X_tr, y_tr = df_train["text"].values, df_train["labels"].values
X_va, y_va = df_val["text"].values,   df_val["labels"].values

CONFIGS = {
    "char_wb (2,4) min_df=5": dict(ngram_range=(2,4), min_df=5),
    "char_wb (1,4) min_df=5": dict(ngram_range=(1,4), min_df=5),
}

MODELS = {
    "MNB":        MultinomialNB(),
    "LogReg":     LogisticRegression(max_iter=1000, n_jobs=-1),
    "LinearSVC":  LinearSVC(),
}

rows = []
for cfg_name, cfg in CONFIGS.items():
    for model_name, model in MODELS.items():
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(analyzer="char_wb", lowercase=True,
                                      sublinear_tf=True, **cfg)),
            ("clf", model),
        ])
        t0 = time.time()
        pipe.fit(X_tr, y_tr)
        fit_t = time.time() - t0
        y_pred = pipe.predict(X_va)
        rows.append(dict(
            config=cfg_name, model=model_name,
            n_features=len(pipe.named_steps["tfidf"].vocabulary_),
            val_acc=accuracy_score(y_va, y_pred),
            val_f1=f1_score(y_va, y_pred, average="macro"),
            fit_sec=round(fit_t, 1),
        ))

import pandas as pd
cmp_df = pd.DataFrame(rows).sort_values("val_f1", ascending=False)
print(cmp_df.to_string(index=False))

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline

C_values = [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
svc_rows = []
for C in C_values:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4),
                                  min_df=5, lowercase=True, sublinear_tf=True)),
        ("clf", LinearSVC(C=C)),
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_va)
    svc_rows.append(dict(C=C,
                         val_acc=accuracy_score(y_va, y_pred),
                         val_f1=f1_score(y_va, y_pred, average="macro")))

svc_df = pd.DataFrame(svc_rows)
print(svc_df.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(svc_df["C"], svc_df["val_f1"], marker="o", label="macro-F1")
plt.xscale("log")
plt.xlabel("C (log scale)")
plt.ylabel("val macro-F1")
plt.title("LinearSVC: качество vs C (char_wb 2-4, min_df=5)")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("svc_C_curve.png", dpi=120)
plt.show()

In [ ]:
logreg_rows = []
for C in [0.1, 1.0, 5.0, 10.0]:
    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2,4),
                                  min_df=5, lowercase=True, sublinear_tf=True)),
        ("clf", LogisticRegression(C=C, max_iter=500, n_jobs=-1)),
    ])
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_va)
    logreg_rows.append(dict(C=C,
                            val_acc=accuracy_score(y_va, y_pred),
                            val_f1=f1_score(y_va, y_pred, average="macro")))

logreg_df = pd.DataFrame(logreg_rows)
print(logreg_df.to_string(index=False))

In [ ]:
FINAL_CONFIG = dict(analyzer="char_wb", ngram_range=(2,4), min_df=5,
                    lowercase=True, sublinear_tf=True)
FINAL_C = 1.0   # подставим лучший C из Шага 2, если отличается

final_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(**FINAL_CONFIG)),
    ("clf", LinearSVC(C=FINAL_C)),
])
final_pipe.fit(X_tr, y_tr)

# Контрольное измерение на val и test
for name, df in [("val", df_val), ("test", df_test)]:
    y_pred = final_pipe.predict(df["text"].values)
    print(f"[{name}] accuracy={accuracy_score(df['labels'], y_pred):.4f}  "
          f"macro-F1={f1_score(df['labels'], y_pred, average='macro'):.4f}")

# Сохраним пайплайн для этапа 6
import pickle, os
with open("final_pipe.pkl", "wb") as f:
    pickle.dump(final_pipe, f)
print("Сохранено: final_pipe.pkl  (размер:", os.path.getsize("final_pipe.pkl") // 1024, "КБ)")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

y_test = df_test["labels"].values
y_pred = final_pipe.predict(df_test["text"].values)

print("=" * 60)
print("Classification report — LinearSVC (1,4) min_df=5, C=1.0")
print("=" * 60)
print(classification_report(y_test, y_pred, digits=4))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

LANGS = ["en", "fr", "de", "es", "it", "ru", "pt", "nl"]

cm = confusion_matrix(y_test, y_pred, labels=LANGS)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=LANGS, yticklabels=LANGS, cbar_kws={"label": "count"})
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title(f"Confusion matrix (test) — accuracy={ (y_test==y_pred).mean():.4f}")
plt.tight_layout()
plt.savefig("confusion_abs.png", dpi=120)
plt.show()

In [ ]:
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(9, 7))
sns.heatmap(cm_norm, annot=True, fmt=".3f", cmap="Blues", vmin=0.98, vmax=1.0,
            xticklabels=LANGS, yticklabels=LANGS, cbar_kws={"label": "recall"})
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Normalized confusion matrix (recall по строкам) — vmin=0.98")
plt.tight_layout()
plt.savefig("confusion_norm.png", dpi=120)
plt.show()

In [ ]:
df_err = df_test.copy()
df_err["pred"] = y_pred
df_err["len_chars"] = df_err["text"].str.len()
errors = df_err[df_err["labels"] != df_err["pred"]][["text", "labels", "pred", "len_chars"]]
errors = errors.sort_values("len_chars").reset_index(drop=True)

print(f"Всего ошибок: {len(errors)} из {len(df_test)} ({100*len(errors)/len(df_test):.2f}%)")
print()
print(errors.to_string(index=False))

In [ ]:
# Сравнение точности на коротких vs длинных текстах
df_test_eval = df_test.copy()
df_test_eval["pred"] = y_pred
df_test_eval["len_chars"] = df_test_eval["text"].str.len()
df_test_eval["correct"] = (df_test_eval["labels"] == df_test_eval["pred"])

bins = [0, 20, 50, 100, 200, 10000]
labels_bins = ["≤20", "21–50", "51–100", "101–200", ">200"]
df_test_eval["len_bin"] = pd.cut(df_test_eval["len_chars"], bins=bins, labels=labels_bins)

print("Точность по длине текста:")
summary = df_test_eval.groupby("len_bin", observed=False).agg(
    n=("correct", "size"),
    accuracy=("correct", "mean"),
).round(4)
print(summary)

In [ ]:
errors.to_csv("errors_test.csv", index=False)
print("Сохранено: errors_test.csv")

In [ ]:
import numpy as np

vec = final_pipe.named_steps["tfidf"]
clf = final_pipe.named_steps["clf"]
feature_names = vec.get_feature_names_out()
classes = clf.classes_

print("Топ-15 n-грамм по весам LinearSVC (char_wb 2-4, min_df=5).")
print("Пробел показан как '·'\n")
for i, lang in enumerate(classes):
    coefs = clf.coef_[i]
    top_idx = np.argsort(coefs)[-15:][::-1]
    pretty = ", ".join(
        f"{feature_names[j].replace(' ', '·')!r}:{coefs[j]:.2f}"
        for j in top_idx
    )
    print(f"{lang}: {pretty}\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

TOP_K = 12
# Собираем матрицу: строки — языки, столбцы — n-граммы
top_all = set()
for i in range(len(classes)):
    coefs = clf.coef_[i]
    top_idx = np.argsort(coefs)[-TOP_K:]
    top_all.update(feature_names[top_idx])
top_all = sorted(top_all)

# Матрица весов
gram_to_idx = {g: np.where(feature_names == g)[0][0] for g in top_all}
W = np.zeros((len(classes), len(top_all)))
for i in range(len(classes)):
    for j, g in enumerate(top_all):
        W[i, j] = clf.coef_[i, gram_to_idx[g]]

plt.figure(figsize=(18, 6))
sns.heatmap(W, xticklabels=[repr(g) for g in top_all], yticklabels=classes,
            cmap="RdBu_r", center=0, cbar_kws={"label": "weight"})
plt.title("LinearSVC: веса n-грамм по языкам (топ-12 на язык)")
plt.xticks(rotation=90, fontsize=9)
plt.tight_layout()
plt.savefig("ngram_weights.png", dpi=120)
plt.show()

In [ ]:
# Простая транслитерация рус→лат (для эксперимента)
TRANSLIT_MAP = {
    'а':'a','б':'b','в':'v','г':'g','д':'d','е':'e','ё':'yo','ж':'zh',
    'з':'z','и':'i','й':'y','к':'k','л':'l','м':'m','н':'n','о':'o',
    'п':'p','р':'r','с':'s','т':'t','у':'u','ф':'f','х':'kh','ц':'ts',
    'ч':'ch','ш':'sh','щ':'shch','ъ':"'",'ы':'y','ь':"'",'э':'e',
    'ю':'yu','я':'ya',
    'А':'A','Б':'B','В':'V','Г':'G','Д':'D','Е':'E','Ё':'Yo','Ж':'Zh',
    'З':'Z','И':'I','Й':'Y','К':'K','Л':'L','М':'M','Н':'N','О':'O',
    'П':'P','Р':'R','С':'S','Т':'T','У':'U','Ф':'F','Х':'Kh','Ц':'Ts',
    'Ч':'Ch','Ш':'Sh','Щ':'Shch','Ъ':"'",'Ы':'Y','Ь':"'",'Э':'E',
    'Ю':'Yu','Я':'Ya',
}

def translit(s):
    return "".join(TRANSLIT_MAP.get(c, c) for c in s)

# Транслитерируем только test ru-тексты, train оставляем как есть
ru_mask = df_test["labels"] == "ru"
test_translit = df_test.copy()
test_translit.loc[ru_mask, "text"] = test_translit.loc[ru_mask, "text"].map(translit)

y_pred_tr = final_pipe.predict(test_translit["text"].values)
ru_pred = y_pred_tr[ru_mask.values]
ru_true = df_test.loc[ru_mask, "labels"].values

print("=== Эффект транслитерации ru→лат на качество распознавания ru ===")
print(f"Оригинал:    accuracy = {(ru_true == final_pipe.predict(df_test.loc[ru_mask,'text'].values)).mean():.4f}")
print(f"Транслит:    accuracy = {(ru_true == ru_pred).mean():.4f}")
print()
from collections import Counter
print("Распределение предсказаний для транслитерированных ru:")
print(Counter(ru_pred).most_common())

In [ ]:
print("=== Сравнение 'сложности' языков: средняя уверенность модели ===")
# Уверенность = margin между 1-м и 2-м классом (для SVM это decision_function)
scores = final_pipe.decision_function(df_test["text"].values)
top1 = scores.max(axis=1)
top2 = np.sort(scores, axis=1)[:, -2]
margin = top1 - top2

df_margin = df_test.copy()
df_margin["margin"] = margin
print("\nСредний margin по языкам:")
print(df_margin.groupby("labels")["margin"].agg(["mean", "median", "min"]).round(3))

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

# Word-level baseline
word_pipe = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="word", ngram_range=(1,2),
                              lowercase=True, sublinear_tf=True, min_df=2)),
    ("clf", LinearSVC(C=1.0)),
])
word_pipe.fit(df_train["text"].values, df_train["labels"].values)

# На полных текстах
y_pred_word = word_pipe.predict(df_test["text"].values)
print(f"Word-level на полных текстах: acc={accuracy_score(df_test['labels'], y_pred_word):.4f}")

In [ ]:
def truncate(texts, n):
    """Обрезаем каждый текст до n символов (включая пробелы)."""
    return [str(t)[:n] for t in texts]

lengths = [5, 10, 15, 20, 30, 50, 100, 200]
y_test = df_test["labels"].values

results = []
for n in lengths:
    X_trunc = truncate(df_test["text"].values, n)

    # char n-gram (final_pipe) — уже обучен, только predict
    y_char = final_pipe.predict(X_trunc)
    acc_char = accuracy_score(y_test, y_char)
    f1_char = f1_score(y_test, y_char, average="macro", zero_division=0)

    # word-level — уже обучен
    y_word = word_pipe.predict(X_trunc)
    acc_word = accuracy_score(y_test, y_word)
    f1_word = f1_score(y_test, y_word, average="macro", zero_division=0)

    results.append(dict(
        length=n,
        char_acc=acc_char, char_f1=f1_char,
        word_acc=acc_word, word_f1=f1_word,
    ))

import pandas as pd
res_len = pd.DataFrame(results)
print(res_len.round(4).to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.plot(res_len["length"], res_len["char_acc"], marker="o", label="char n-grams (1,4)", linewidth=2)
plt.plot(res_len["length"], res_len["word_acc"], marker="s", label="word n-grams (1,2)", linewidth=2)
plt.xscale("log")
plt.axvline(20, color="gray", linestyle="--", alpha=0.5, label="порог «короткий текст»")
plt.xlabel("длина текста (символов, log scale)")
plt.ylabel("accuracy на test")
plt.title("Устойчивость к длине: char n-gram vs word-level")
plt.legend()
plt.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.savefig("length_robustness.png", dpi=120)
plt.show()

In [ ]:
import random
random.seed(42)

def add_typos(text, p_typo=0.1):
    """С вероятностью p на каждый символ — заменяем/удаляем/переставляем."""
    text = str(text)
    chars = list(text)
    out = []
    i = 0
    while i < len(chars):
        r = random.random()
        if r < p_typo / 3 and i + 1 < len(chars):
            # перестановка
            out.append(chars[i+1]); out.append(chars[i]); i += 2
        elif r < 2*p_typo/3:
            # замена на случайный символ
            out.append(random.choice("abcdefghijklmnopqrstuvwxyzàâéèêëîïôùûüçñ"))
            i += 1
        elif r < p_typo:
            # удаление
            i += 1
        else:
            out.append(chars[i]); i += 1
    return "".join(out)

typo_rates = [0.0, 0.05, 0.10, 0.20, 0.30, 0.40, 0.50]
results_typo = []
y_test = df_test["labels"].values

for p in typo_rates:
    X_typo = [add_typos(t, p_typo=p) for t in df_test["text"].values]
    y_char = final_pipe.predict(X_typo)
    y_word = word_pipe.predict(X_typo)
    results_typo.append(dict(
        p_typo=p,
        char_acc=accuracy_score(y_test, y_char),
        word_acc=accuracy_score(y_test, y_word),
    ))

res_typo = pd.DataFrame(results_typo)
print(res_typo.round(4).to_string(index=False))

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(res_typo["p_typo"]*100, res_typo["char_acc"], marker="o", label="char n-grams", linewidth=2)
plt.plot(res_typo["p_typo"]*100, res_typo["word_acc"], marker="s", label="word n-grams", linewidth=2)
plt.xlabel("доля опечаток (%)")
plt.ylabel("accuracy на test")
plt.title("Устойчивость к опечаткам: char n-gram vs word-level")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("typo_robustness.png", dpi=120)
plt.show()

In [ ]:
print("Пример деградации word-level на 10-символьных текстах:")
samples_10 = truncate(df_test["text"].values[:20], 10)
print(f"{'текст':<15} {'true':<5} {'char':<5} {'word':<5}")
for s, true in zip(samples_10, df_test["labels"].values[:20]):
    pc = final_pipe.predict([s])[0]
    pw = word_pipe.predict([s])[0]
    mark_c = "✓" if pc == true else "✗"
    mark_w = "✓" if pw == true else "✗"
    print(f"{s!r:<15} {true:<5} {pc} {mark_c}   {pw} {mark_w}")

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {"Сценарий": "Полные тексты (test)", "char n-gram": 0.9990, "word n-gram": 0.9952, "Разрыв (п.п.)": 0.38},
    {"Сценарий": "Обрезка до 50 симв.", "char n-gram": 0.9982, "word n-gram": 0.9900, "Разрыв (п.п.)": 0.82},
    {"Сценарий": "Обрезка до 20 симв.", "char n-gram": 0.9745, "word n-gram": 0.9268, "Разрыв (п.п.)": 4.77},
    {"Сценарий": "Обрезка до 10 симв.", "char n-gram": 0.8960, "word n-gram": 0.7917, "Разрыв (п.п.)": 10.43},
    {"Сценарий": "Обрезка до 5 симв.",  "char n-gram": 0.7688, "word n-gram": 0.6378, "Разрыв (п.п.)": 13.10},
    {"Сценарий": "Опечатки 10%",       "char n-gram": 0.9942, "word n-gram": 0.9448, "Разрыв (п.п.)": 4.94},
    {"Сценарий": "Опечатки 30%",       "char n-gram": 0.9442, "word n-gram": 0.6798, "Разрыв (п.п.)": 26.44},
    {"Сценарий": "Опечатки 50%",       "char n-gram": 0.7042, "word n-gram": 0.3418, "Разрыв (п.п.)": 36.24},
])
print(summary.to_string(index=False))